In [2]:
sentences = [
    '나는 오늘 밥을 먹었다',
    '나는 오늘 영화를 보았다'
]
# 공백으로 분류
words_list = [sentence.split() for sentence in sentences]
words_list

[['나는', '오늘', '밥을', '먹었다'], ['나는', '오늘', '영화를', '보았다']]

In [3]:
next_words = []
for words in words_list:
    for i in range(len(words)-2):
        if words[i] == '나는' and words[i+1] == '오늘':
            next_words.append(words[i+2])
print(f'나는 오늘 다음에 등장한 단어들: {next_words}')

나는 오늘 다음에 등장한 단어들: ['밥을', '영화를']


In [4]:
# 말뭉치 특수 토큰 부착 / 기본 토큰화
raw_corpus = [
    '나는 오늘 파이토치 학습을 한다',
    '나는 내일 자연어 처리를 한다',
    '오늘 파이토치 학습은 매우 즐겁다',
    '파이토치 코딩은 매우 재미있다'
]
# BOS 부착함수
def process_corpus(corpus):
    processed = []
    for sentence in corpus:
        processed.append(['<s>'] + sentence.split() + ['</s>'])
    return processed

tokenized_corpus = process_corpus(raw_corpus)
tokenized_corpus

[['<s>', '나는', '오늘', '파이토치', '학습을', '한다', '</s>'],
 ['<s>', '나는', '내일', '자연어', '처리를', '한다', '</s>'],
 ['<s>', '오늘', '파이토치', '학습은', '매우', '즐겁다', '</s>'],
 ['<s>', '파이토치', '코딩은', '매우', '재미있다', '</s>']]

In [6]:
# n_gram 추출기 구현 / 데이터 베이스화
from collections import Counter
def extract_ngrams(tokenized_corpus, n):
    ngram_counts = Counter()
    for tokens in tokenized_corpus:
        # 토큰길이가 N보다 작으면 추출불가
        if len(tokens) < n:
            continue
        # 슬라이딩 윈도우 방식
        for i in range(len(tokens) - n+1):
            ngram = tuple(tokens[i : i+n])
            ngram_counts[ngram] += 1
    return ngram_counts


In [19]:
# unigram(1-gram) 추출
unigram = extract_ngrams(tokenized_corpus,1)
bigram = extract_ngrams(tokenized_corpus, 2)
trigram = extract_ngrams(tokenized_corpus, 3)
forgram = extract_ngrams(tokenized_corpus, 4)

In [20]:
unigram.most_common(3), bigram.most_common(3), trigram.most_common(3), forgram.most_common(3)

([(('<s>',), 4), (('</s>',), 4), (('파이토치',), 3)],
 [(('<s>', '나는'), 2), (('오늘', '파이토치'), 2), (('한다', '</s>'), 2)],
 [(('<s>', '나는', '오늘'), 1),
  (('나는', '오늘', '파이토치'), 1),
  (('오늘', '파이토치', '학습을'), 1)],
 [(('<s>', '나는', '오늘', '파이토치'), 1),
  (('나는', '오늘', '파이토치', '학습을'), 1),
  (('오늘', '파이토치', '학습을', '한다'), 1)])

In [22]:
def simulate_prediction(ngram_counts, context_words):
    # context_words = 뒤에 올 단어 예측 후보
    context_len = len(context_words)
    found = False
    # 튜플의 앞부분이 context_words와 일치하는 n_garm을 탐색
    for ngram, count in ngram_counts.items():
        if list(ngram[:context_len]) == context_words:
            next_word = ngram[context_len]
            print(f'-> [예측단어: {next_word}] [등장횟수: {count}번]')
            found = True
    if not found:
        print(f'-> 데이터베이스에 해당 문맥이 존재하지 않습니다 (희소문제 발생)')


In [23]:
# 나는 다음에 올 단어 예측
simulate_prediction(bigram, ['나는'])

-> [예측단어: 오늘] [등장횟수: 1번]
-> [예측단어: 내일] [등장횟수: 1번]


In [24]:
# trigram
simulate_prediction(trigram, ['나는', '오늘'])

-> [예측단어: 파이토치] [등장횟수: 1번]


In [25]:
# 희소문제 데이터에 없는 패턴 테스트
simulate_prediction(trigram, ['파이토치', '처리를'])

-> 데이터베이스에 해당 문맥이 존재하지 않습니다 (희소문제 발생)


In [26]:
simulate_prediction(forgram, ['나는', '오늘', '파이토치'])

-> [예측단어: 학습을] [등장횟수: 1번]
